In [1]:
library(data.table)
library(ggplot2)
library(openxlsx)

In [ ]:
project_dir <- <path to project>

In [5]:
cov_thrs <- list("human_cava_cohort_plasma_1" = 0.8, "human_cava_cohort_plasma_2" = 0.8, 
                 "human_bike_cohort_plasma_rerun" = 0.75, "human_bike_cohort_plaque_aorta_rerun" = 0.75)

In [7]:
full_summary_df <- data.table()
for(name_id in names(cov_thrs)){
    CORTHR = cov_thrs[name_id]

    summary_df <- fread(file.path(project_dir, name_id, "aggregated", "cv_comparison.csv"))

    summary_df[, status:="FAIL",]
    summary_df[summary_df$cor_coef > CORTHR, status:="PASS",]
    one_lib_failed <- unique(summary_df[status=="FAIL"]$sample_name)

    ## if one library failed, other also:
    summary_df[summary_df$sample_name %in% one_lib_failed,]$status <- "FAIL"
    summary_df <- unique(summary_df[, c("sample_name","cor_coef", "library", "status")])

    summary_df$run_id <- name_id

    full_summary_df <- rbind(summary_df, full_summary_df)
    
}


In [10]:
write.csv(full_summary_df, file.path(project_dir, "correlation_replicas.csv"))